# Supervised Fine-Tuning with HuggingFace TRL (LoRA & QLoRA)

**TRL** (Transformer Reinforcement Learning) is HuggingFace's library for
fine-tuning language models. Its `SFTTrainer` is the standard starting point
for supervised instruction-tuning.

This notebook covers:
- **Section A — Standard LoRA:** float16 base model + LoRA adapters
- **Section B — QLoRA:** 4-bit NF4 quantized base model + LoRA adapters
- Best practices for both, with gotchas called out throughout

**Model used:** `facebook/opt-125m` (125M params, no gating, fits any GPU)

| | LoRA | QLoRA |
|-|------|-------|
| Base model VRAM | ~0.5 GB | ~0.1 GB |
| Quality | Baseline | ~2–5% lower |
| Extra deps | peft, accelerate | + bitsandbytes |

> **GPU recommended.** OPT-125M can train on CPU but will be very slow.

## Environment Setup

Run once in a terminal before opening this notebook:

```bash
cd projects/huggingface_trl
uv sync --no-install-workspace
uv run ipython kernel install --user --env VIRTUAL_ENV ../../.venv --name=huggingface_trl
```

Select the **`huggingface_trl`** kernel in JupyterLab.

In [ ]:
import sys
import torch
import transformers, trl, peft, datasets

print(f'Python {sys.version}')
print(f'PyTorch   {torch.__version__}')
print(f'CUDA      {torch.cuda.is_available()}')
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f'GPU       {props.name}  VRAM: {props.total_memory / 1e9:.1f} GB')
print(f'transformers {transformers.__version__}  trl {trl.__version__}')
print(f'peft {peft.__version__}  datasets {datasets.__version__}')

In [ ]:
import sys, pathlib, os
sys.path.insert(0, str(pathlib.Path('../../../').resolve()))
from dotenv_config import dot_env_settings

if dot_env_settings.HF_TOKEN:
    os.environ['HF_TOKEN'] = dot_env_settings.HF_TOKEN
    print('HF_TOKEN loaded.')
else:
    print('WARNING: No HF_TOKEN. Public models only.')

## Tokenizer Loading and the Pad Token Gotcha

> **Gotcha #1 — GPT-style models have no pad token.**
> OPT, GPT-2, LLaMA (base, not chat) are trained with no padding. Their
> tokenizers have `pad_token = None`. The moment `SFTTrainer` tries to pad
> a batch it crashes with:
> ```
> ValueError: Asking to pad but the tokenizer does not have a padding token.
> ```
> Fix: `tokenizer.pad_token = tokenizer.eos_token` **before** passing to the
> trainer. Also set `padding_side = 'right'` for causal LM training so the
> padding is appended after the sequence, not before.

In [ ]:
from transformers import AutoTokenizer

MODEL_NAME = 'facebook/opt-125m'
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print(f'pad_token before fix: {tokenizer.pad_token!r}')  # None

tokenizer.pad_token = tokenizer.eos_token     # reuse EOS as pad
tokenizer.padding_side = 'right'              # pad on the right for causal LMs

print(f'pad_token after fix:  {tokenizer.pad_token!r}')

## Dataset Preparation

We use a small inline dataset so this notebook runs without downloading
anything extra. The format is Alpaca-style: instruction + response.

> **Gotcha #2 — Chat template mismatch.**
> Some tokenizers ship with a `chat_template`. If present, TRL may try to
> apply it. For OPT (no template) and other base models, use an explicit
> format string instead of `tokenizer.apply_chat_template()` to avoid
> unexpected `<s>` / `[INST]` tokens appearing in your data.

In [ ]:
from datasets import Dataset

ALPACA_PROMPT = (
    'Below is an instruction. Write a response that appropriately completes it.\n\n'
    '### Instruction:\n{instruction}\n\n'
    '### Response:\n{response}'
)

raw = [
    {'instruction': 'What is the capital of France?', 'response': 'The capital of France is Paris.'},
    {'instruction': 'Write a haiku about autumn.', 'response': 'Leaves fall gently down / Crisp air whispers through bare trees / Winter waits nearby'},
    {'instruction': 'Explain what an API is.', 'response': 'An API (Application Programming Interface) lets software components communicate with each other through defined rules.'},
    {'instruction': 'What is 17 multiplied by 6?', 'response': '17 multiplied by 6 is 102.'},
    {'instruction': 'Write a Python one-liner to flatten a list of lists.', 'response': 'flat = [x for sublist in nested for x in sublist]'},
    {'instruction': 'What causes rainbows?', 'response': 'Rainbows are caused by sunlight refracting, dispersing, and reflecting inside water droplets, separating white light into its spectrum.'},
    {'instruction': 'Summarise the plot of Romeo and Juliet in two sentences.', 'response': 'Two teenagers from rival families fall in love and secretly marry. Their families\u2019 feud ultimately leads to both their deaths.'},
    {'instruction': 'Convert 100 kilometres to miles.', 'response': '100 kilometres is approximately 62.14 miles.'},
    {'instruction': 'Give a tip for improving focus while studying.', 'response': 'Use the Pomodoro technique: work for 25 minutes, then take a 5-minute break.'},
    {'instruction': 'What is the boiling point of water at sea level?', 'response': 'Water boils at 100\u00b0C (212\u00b0F) at standard sea-level atmospheric pressure.'},
]

def format_row(row):
    return {'text': ALPACA_PROMPT.format(**row)}

dataset = Dataset.from_list(raw).map(format_row)
print(dataset)
print('\nSample:')
print(dataset[0]['text'])

---
## Section A — Standard LoRA

Load the model in **float16** and inject LoRA adapters. This is the default
approach when VRAM is not the primary constraint.

In [ ]:
from transformers import AutoModelForCausalLM

model_lora = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map='auto',
)
model_lora.config.pad_token_id = tokenizer.pad_token_id

# MUST disable KV cache when using gradient checkpointing with PEFT
# Gotcha #3: leaving use_cache=True causes a crash when gradient_checkpointing=True
model_lora.config.use_cache = False

params = sum(p.numel() for p in model_lora.parameters())
print(f'Loaded {MODEL_NAME}  ({params/1e6:.0f}M params, {model_lora.dtype})')

### LoRA Configuration

Key parameters:
- **`r`** — rank (4–64 typical). Higher = more expressive but more VRAM.
- **`lora_alpha`** — scaling factor. Rule of thumb: `alpha = 2 * r`.
- **`target_modules`** — which weight matrices to inject into. OPT uses
  `q_proj` and `v_proj`.
- **`bias='none'`** — don't train bias terms (saves memory, works well).

In [ ]:
from peft import LoraConfig, get_peft_model, TaskType

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=['q_proj', 'v_proj'],
    lora_dropout=0.05,
    bias='none',
    task_type=TaskType.CAUSAL_LM,
)

model_lora = get_peft_model(model_lora, lora_config)
model_lora.print_trainable_parameters()
# Expect ~0.1% trainable — that is the power of LoRA

### Train with SFTTrainer

> **Gotcha #4 — `packing=True` can silently break training.**
> Packing concatenates examples end-to-end for efficiency, but requires
> every example to end with an EOS token. If EOS is missing, examples
> bleed into each other and the model learns to ignore boundaries.
> Start with `packing=False`. Enable it only once you've verified your
> data has correct EOS tokens.

In [ ]:
from trl import SFTTrainer, SFTConfig

sft_config_lora = SFTConfig(
    output_dir='./trl_opt125m_lora_output',
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,    # effective batch = 8
    gradient_checkpointing=True,      # saves VRAM; requires use_cache=False
    learning_rate=2e-4,
    lr_scheduler_type='cosine',
    warmup_ratio=0.03,
    fp16=torch.cuda.is_available(),
    logging_steps=10,
    save_strategy='epoch',
    max_seq_length=256,
    dataset_text_field='text',
    packing=False,                    # see gotcha note above
    report_to='none',
)

trainer_lora = SFTTrainer(
    model=model_lora,
    args=sft_config_lora,
    train_dataset=dataset,
    tokenizer=tokenizer,
)

print('Training (LoRA)...')
trainer_lora.train()
print('Done.')

In [ ]:
LORA_SAVE_PATH = './trl_opt125m_lora_adapter'
trainer_lora.save_model(LORA_SAVE_PATH)
tokenizer.save_pretrained(LORA_SAVE_PATH)
print(f'LoRA adapter saved to {LORA_SAVE_PATH}')

# Free VRAM before Section B
del model_lora, trainer_lora
import gc; gc.collect()
if torch.cuda.is_available(): torch.cuda.empty_cache()

---
## Section B — QLoRA

**QLoRA** = 4-bit NF4-quantized base model + float16 LoRA adapters.
Introduced in the [QLoRA paper (Dettmers et al., 2023)](https://arxiv.org/abs/2305.14314).

The base model is loaded in 4-bit, cutting its VRAM footprint by ~4×.
The LoRA adapter weights remain in float16, so the actual gradient updates
happen in higher precision.

### Step 1 — Configure 4-bit Quantization

> **Gotcha #5 — Use `bnb_4bit_quant_type='nf4'`, not `'fp4'`.**
> NF4 (NormalFloat4) is the data type described in the QLoRA paper. It is
> designed for normally-distributed weights and gives significantly better
> quality than FP4. Always use `nf4` unless you have a specific reason.

> **Gotcha #6 — `bnb_4bit_use_double_quant=True` for extra VRAM savings.**
> Double quantisation quantises the quantisation constants themselves, saving
> an extra ~0.4 bits per parameter at negligible quality cost.

In [ ]:
from transformers import AutoModelForCausalLM, BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',            # NormalFloat4 from QLoRA paper
    bnb_4bit_use_double_quant=True,       # quantise the quant constants too
    bnb_4bit_compute_dtype=torch.bfloat16,  # computation dtype (not storage)
)

model_qlora = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map='auto',
)
model_qlora.config.pad_token_id = tokenizer.pad_token_id
model_qlora.config.use_cache = False

print(f'Model dtype: {model_qlora.dtype}')  # float32 wrapper, 4-bit storage
mem = sum(p.numel() * p.element_size() for p in model_qlora.parameters())
print(f'Approximate parameter memory: {mem / 1e6:.1f} MB')

### Step 2 — Prepare the Quantized Model for Training

> **Gotcha #7 — `prepare_model_for_kbit_training` is mandatory for QLoRA.**
> This PEFT helper:
> 1. Casts layer norms to float32 so gradients are numerically stable
> 2. Enables gradient computation through the quantized layers
> 3. Enables gradient checkpointing safely
>
> Skipping it means **gradients will not flow** through the base model
> layers. Training will appear to run but the loss will barely decrease.

In [ ]:
from peft import prepare_model_for_kbit_training, LoraConfig, get_peft_model, TaskType

# MUST come before get_peft_model for QLoRA
model_qlora = prepare_model_for_kbit_training(model_qlora)

lora_config_q = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=['q_proj', 'v_proj'],
    lora_dropout=0.05,
    bias='none',
    task_type=TaskType.CAUSAL_LM,
)

model_qlora = get_peft_model(model_qlora, lora_config_q)
model_qlora.print_trainable_parameters()
# Same trainable param count as LoRA — but base model uses ~4x less VRAM

In [ ]:
sft_config_qlora = SFTConfig(
    output_dir='./trl_opt125m_qlora_output',
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    gradient_checkpointing=True,
    learning_rate=2e-4,
    lr_scheduler_type='cosine',
    warmup_ratio=0.03,
    # For QLoRA use bf16 if available; fp16 can cause overflow with 4-bit models
    bf16=torch.cuda.is_bf16_supported(),
    fp16=not torch.cuda.is_bf16_supported() and torch.cuda.is_available(),
    logging_steps=10,
    save_strategy='epoch',
    max_seq_length=256,
    dataset_text_field='text',
    packing=False,
    report_to='none',
)

trainer_qlora = SFTTrainer(
    model=model_qlora,
    args=sft_config_qlora,
    train_dataset=dataset,
    tokenizer=tokenizer,
)

print('Training (QLoRA)...')
trainer_qlora.train()
print('Done.')

In [ ]:
QLORA_SAVE_PATH = './trl_opt125m_qlora_adapter'
trainer_qlora.save_model(QLORA_SAVE_PATH)
tokenizer.save_pretrained(QLORA_SAVE_PATH)
print(f'QLoRA adapter saved to {QLORA_SAVE_PATH}')

## Inference with a Saved Adapter

> **Gotcha #8 — Re-enable KV cache for inference.**
> We set `model.config.use_cache = False` for training. At inference time
> the KV cache makes generation much faster. Always re-enable it on the
> loaded inference model.

In [ ]:
from peft import PeftModel

# Load fresh base model + adapter for inference
base = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=torch.float16, device_map='auto')
base.config.use_cache = True   # re-enable for inference
ft_model = PeftModel.from_pretrained(base, QLORA_SAVE_PATH)
ft_model.eval()

prompt = ALPACA_PROMPT.format(
    instruction='What is the capital of Germany?',
    response='',
)
inputs = tokenizer(prompt, return_tensors='pt').to(ft_model.device)

with torch.no_grad():
    output = ft_model.generate(
        **inputs,
        max_new_tokens=60,
        temperature=0.7,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id,
    )
print(tokenizer.decode(output[0], skip_special_tokens=True))

## Merging the Adapter into the Base Model (Optional)

For deployment you often want a single standalone model without the
adapter indirection. `merge_and_unload()` folds the LoRA weights back
into the base model.

> **Gotcha #9 — QLoRA merge dequantizes.**
> If you loaded the base model in 4-bit, merging dequantizes it to float16
> first. The resulting merged model will be float16, **not** 4-bit.
> For OPT-125M this is ~240 MB; for a 7B model it is ~14 GB.

In [ ]:
# Uncomment to merge and save a standalone model
# merged = ft_model.merge_and_unload()
# merged.save_pretrained('./trl_opt125m_merged')
# tokenizer.save_pretrained('./trl_opt125m_merged')
# print('Merged model saved.')
print('Uncomment the lines above to merge the adapter into the base model.')

## Summary

### Gotcha recap

| # | Gotcha | Fix |
|---|--------|-----|
| 1 | GPT/OPT has no `pad_token` | `tokenizer.pad_token = tokenizer.eos_token` |
| 2 | Chat template mismatch | Use explicit format string, not `apply_chat_template` |
| 3 | `use_cache=True` + gradient checkpointing | Set `model.config.use_cache = False` before training |
| 4 | `packing=True` without EOS tokens | Start with `packing=False` |
| 5 | Using `fp4` instead of `nf4` | Always `bnb_4bit_quant_type='nf4'` for QLoRA |
| 6 | Skipping double quantisation | Set `bnb_4bit_use_double_quant=True` for free VRAM |
| 7 | Missing `prepare_model_for_kbit_training` | Call it **before** `get_peft_model` in QLoRA |
| 8 | `use_cache=False` at inference | Set `model.config.use_cache = True` on inference model |
| 9 | QLoRA merge disk budget | Dequantizes to float16 — plan accordingly |

### LoRA vs QLoRA tradeoff

| | LoRA | QLoRA |
|-|------|-------|
| Base VRAM (125M model) | ~0.5 GB | ~0.1 GB |
| Extra setup step | — | `prepare_model_for_kbit_training` |
| Merge produces | float16 | float16 (dequantized) |
| When to use | VRAM not a concern | VRAM-constrained, large models |

**Next steps:** swap `facebook/opt-125m` for `meta-llama/Llama-3.2-1B` (needs
HF token), add `report_to='mlflow'` for experiment tracking, try
`DPOTrainer` for preference alignment.